In [1]:
# Import the libraries required for text processing, 
# dimensionality reduction, modeling, and evaluation

import pandas as pd
import re
import nltk
import numpy as np
from nltk.corpus import wordnet
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier
from sklearn.ensemble import GradientBoostingClassifier

# Download NLTK resources for tokenization and lemmatization

nltk.download('punkt')
nltk.download('wordnet')
nltk.download('averaged_perceptron_tagger')

# Auxiliary functions for text preprocessing

# Remove emojis from text using a regular expression that covers Unicode emoji ranges
def remove_emojis(text):
    if isinstance(text, str):
        pattern = re.compile("[" 
            "\U0001F600-\U0001F64F" "\U0001F300-\U0001F5FF"
            "\U0001F680-\U0001F6FF" "\U0001F700-\U0001F77F"
            "\U0001F780-\U0001F7FF" "\U0001F800-\U0001F8FF"
            "\U0001F900-\U0001F9FF" "\U0001FA00-\U0001FA6F"
            "\U0001FA70-\U0001FAFF" "\U00002702-\U000027B0"
            "\U000024C2-\U0001F251" "]+", flags=re.UNICODE)
        return pattern.sub(r'', text)
    return text

# Initialize the WordNet lemmatizer
lemmatizer = WordNetLemmatizer()

# Obtain the grammatical category (POS) of a word, necessary for accurate lemmatization
def get_pos_tag(word):
    tag = nltk.pos_tag([word])[0][1][0].upper()
    tag_type = {"J": wordnet.ADJ, "N": wordnet.NOUN, "V": wordnet.VERB, "R": wordnet.ADV}
    return tag_type.get(tag, wordnet.NOUN)

# Full text lemmatization function (tokenization + lemmatization)
def lemmatize_text(text):
    words = nltk.word_tokenize(text)
    lemmatized_words = [lemmatizer.lemmatize(w, get_pos_tag(w)) for w in words]
    return ' '.join(lemmatized_words)

# Load and preprocess the dataset


url = "https://raw.githubusercontent.com/justmarkham/pycon-2016-tutorial/master/data/sms.tsv"
df = pd.read_csv(url, sep='\t', header=None, names=['label', 'message'])

# Convert text labels to binary values: 'ham' → 0, 'spam' → 1
df['label'] = df['label'].replace({'ham': 0, 'spam': 1})

# Apply text cleaning: emoji removal and lemmatization
df['message'] = df['message'].apply(remove_emojis).apply(lemmatize_text)

# Define stopwords

stopwords = ['a', 'an', 'the', 'in', 'on', 'at', 'to', 'of', 'and', 'or',
            'is', 'it', 'for', 'with', 'that', 'this', 'as', 'was', 'be',
            'are', 'were', 'been', 'from', 'by', 'about', 'into', 'out',
            'up', 'down', 'over', 'under', 'then', 'than', 'so', 'but', 'not']

# Define input and output variables

X = df["message"]
y = df["label"]

# Define the list of components (k) for SVD reduction

k_values = [100]  # number of components to test for SVD
results = []

In [2]:
i=0
from sklearn.ensemble import GradientBoostingClassifier

# Configure stratified cross-validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Main loop: for each value of k, train and evaluate the model
for k in k_values:
    scores = []
    for train_idx, test_idx in cv.split(X, y):
        # Split the text into training and test sets
        X_train_raw, X_test_raw = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        # TF-IDF vectorization fitted exclusively on the training set
        tfidf = TfidfVectorizer(max_features=10000, stop_words=stopwords)
        X_train_tfidf = tfidf.fit_transform(X_train_raw)
        X_test_tfidf = tfidf.transform(X_test_raw)
        # SVD dimensionality reduction fitted only on the training set
        svd = TruncatedSVD(n_components=k, random_state=42)
        X_train_svd = svd.fit_transform(X_train_tfidf)
        X_test_svd = svd.transform(X_test_tfidf)
        # Train the XGBoost model
        model = XGBClassifier(
            max_depth=6,
            n_estimators=500,
            learning_rate=0.01,
            random_state=42
        )
        model.fit(X_train_svd, y_train)
        print("Iteration: "+str(i) )  # Print progress information for the current iteration

        # Compute AUC-ROC on the test set
        probs = model.predict_proba(X_test_svd)[:, 1]
        auc = roc_auc_score(y_test, probs)
        scores.append(auc)
        print("AUC-ROC: Fold " + str(i) +  " : " + str(auc))  # Print the AUC score for the current fold
        i=i+1
    # Store the mean result for this value of k
    results.append({
        'k_components': k,
        'mean_auc_roc': np.mean(scores)
    })

# Store the results
df_results = pd.DataFrame(results)


Iteration: 0
AUC-ROC: Fold 0 : 0.9848497409326425
Iteration: 1
AUC-ROC: Fold 1 : 0.9810224525043177
Iteration: 2
AUC-ROC: Fold 2 : 0.9892756546232223
Iteration: 3
AUC-ROC: Fold 3 : 0.9658552700212122
Iteration: 4
AUC-ROC: Fold 4 : 0.9878916437736899


In [3]:
# Organize the results
df_results = pd.DataFrame(results)

# Display the results table
print("\n Cross-validation results (mean AUC-ROC for k=100):\n")
print(df_results.to_string(index=False, float_format="%.4f"))


 Cross-validation results (mean AUC-ROC for k=100):

 k_components  mean_auc_roc
          100        0.9818
